# Topic 14 — Support Vector Machines (SVM)
### Theory → tiny example → margin visualization → kernels → C & gamma → text classification.

SVM finds the **hyperplane** (a line in 2D, a plane in 3D, a higher-dim surface beyond that) that
separates two classes with the **widest possible margin** — the gap between the hyperplane and the
closest points of each class. Those closest points are called **support vectors**; they're the only
points that actually determine where the boundary sits (all other points could move without
changing the boundary at all).

SVMs are historically one of the strongest classical algorithms for high-dimensional sparse data
like TF-IDF text vectors (Topic 24) — very relevant for your paper.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC, LinearSVC
from sklearn.datasets import make_classification, make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

rng = np.random.default_rng(0)

## 1. Hyperplane, margin, support vectors — visualized

In [ ]:
X, y = make_classification(
    n_samples=40, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=2.5, random_state=7
)

clf = SVC(kernel="linear", C=1.0)
clf.fit(X, y)

# Plot data, decision boundary, margins, and support vectors
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k")
plt.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="black")
# The middle solid line (level 0) is the hyperplane; the dashed lines (levels -1, +1) mark the margin
plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
            s=150, facecolors="none", edgecolors="green", linewidths=2, label="support vectors")
plt.legend()
plt.title("SVM: hyperplane (solid), margins (dashed), support vectors (circled)")
plt.show()
print("number of support vectors:", len(clf.support_vectors_), "out of", len(X), "total points")
# Only the circled points determine the boundary -- move any OTHER point and the line doesn't change.

## 2. The `C` parameter — margin width vs classification error tradeoff

`C` controls how much the SVM penalizes misclassified points:
- **Small C**: wider margin, tolerates some points inside the margin or misclassified (softer boundary).
- **Large C**: narrower margin, tries hard to classify every training point correctly (can overfit).

In [ ]:
# Add some overlap/noise to make the C tradeoff visible
X_noisy, y_noisy = make_classification(
    n_samples=60, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=0.8, flip_y=0.05, random_state=7
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, C in zip(axes, [0.01, 1, 100]):
    c = SVC(kernel="linear", C=C).fit(X_noisy, y_noisy)
    xx, yy = np.meshgrid(np.linspace(X_noisy[:,0].min()-1, X_noisy[:,0].max()+1, 200),
                          np.linspace(X_noisy[:,1].min()-1, X_noisy[:,1].max()+1, 200))
    Z = c.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="black")
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1], c=y_noisy, cmap="bwr", edgecolor="k")
    ax.set_title(f"C={C}  (n_support_vectors={c.n_support_.sum()})")
plt.tight_layout()
plt.show()
# Small C -> wide margin, more support vectors, tolerates errors.
# Large C -> narrow margin, fewer support vectors, fits training data more tightly (risk of overfitting).

## 3. Kernels: when data isn't linearly separable

A **kernel** lets SVM draw *non-linear* boundaries by implicitly mapping data into a higher-dimensional
space where it BECOMES linearly separable — without ever explicitly computing that mapping (the "kernel trick").

- **Linear kernel**: a straight line/plane — use when classes are roughly linearly separable (common for text).
- **RBF kernel** (radial basis function): can draw curved, even closed-loop boundaries.

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, kernel in zip(axes, ["linear", "rbf"]):
    c = SVC(kernel=kernel, C=1.0).fit(X_circ, y_circ)
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
    Z = c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="bwr")
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap="bwr", edgecolor="k")
    acc = accuracy_score(y_circ, c.predict(X_circ))
    ax.set_title(f"kernel={kernel}  (train acc={acc:.2f})")
plt.tight_layout()
plt.show()
# Linear kernel fails completely on this ring-shaped data (a straight line can't separate a ring from its center).
# RBF succeeds by drawing a curved boundary.

## 4. `gamma` — how far a single point's influence reaches (RBF kernel only)

- **Low gamma**: each point's influence reaches far, giving a smoother boundary.
- **High gamma**: each point's influence is very local, giving a tighter, more wiggly boundary (overfit risk).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, gamma in zip(axes, [0.1, 1, 20]):
    c = SVC(kernel="rbf", C=1.0, gamma=gamma).fit(X_circ, y_circ)
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
    Z = c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="bwr")
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap="bwr", edgecolor="k")
    ax.set_title(f"gamma={gamma}")
plt.tight_layout()
plt.show()

## 5. SVM for text classification

For text, `LinearSVC` (an optimized linear-kernel SVM) is a standard strong baseline —
usually preferred over `SVC(kernel="linear")` because it scales much better to high-dimensional,
sparse TF-IDF/BoW features (thousands of vocabulary "features").

In [ ]:
texts = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "great job today team", "have a wonderful day",
    "you are an idiot", "nice work everyone", "thanks for your help",
    "you should just disappear", "well done on the project", "excellent effort today",
]
y_texts = np.array([1,1,1,1, 0,0, 1,0,0, 1,0,0])

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, y_texts, test_size=0.3, random_state=42, stratify=y_texts
)

vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train_txt)
X_test_vec = vec.transform(X_test_txt)

svm_text = LinearSVC()
svm_text.fit(X_train_vec, y_train)

print("test accuracy:", accuracy_score(y_test, svm_text.predict(X_test_vec)))
print("predictions:", svm_text.predict(X_test_vec))
print("true labels:", y_test)

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Re-run the C-tradeoff plot with class_sep=1.5 (less overlap) -- does the effect of C look weaker?
# 2. Try kernel="poly" with degree=3 on the circles dataset and compare its boundary to rbf.
# 3. On the text example, swap TfidfVectorizer for CountVectorizer -- does accuracy change?
# 4. Read sklearn docs for LinearSVC's `C` parameter -- does increasing it seem to help or hurt
#    on this tiny toy text dataset? (With only 12 examples, don't over-interpret the result --
#    the point is to practice changing hyperparameters and observing effects.)

---
### Next up: **Topic 15 — Feature Scaling**.

Say "next" when you're ready.